Below is **a complete MCP Server + MCP Client setup** that wraps your Adidas RAG pipeline as an **MCP Tool** so any MCP-compatible client (Claude Desktop, VS Code MCP extension, LangChain agent, etc.) can query the vector database.

I’m giving you:

### ✅ `adidas_rag_server.py` — MCP Server exposing your RAG search

### ✅ `adidas_rag_client.py` — MCP Client/Agent that calls the server

### ✅ Clear folder structure

### ✅ How to run

---

# ✅ Folder Structure

```
your_project/
│
├── adidas_rag/
│   ├── rag_pipeline.py          # your original code (modularized)
│
├── mcp/
│   ├── adidas_rag_server.py     # MCP server exposing search tool
│   ├── adidas_rag_client.py     # MCP client/agent
│
└── data/
    └── adidas.csv
```

---

# ✅ 1. `rag_pipeline.py`

(Rewritten slightly into importable functions)

```python
import os
import pandas as pd
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

load_dotenv()

def load_csv_as_documents(csv_path: str):
    df = pd.read_csv(csv_path)

    documents = []
    for idx, row in df.iterrows():
        text = " | ".join([f"{col}: {str(row[col])}" for col in df.columns])

        doc = Document(
            page_content=text,
            metadata={col: row[col] for col in df.columns}
        )
        documents.append(doc)

    return documents


def build_vector_db(documents, persist_dir="db/adidas_products"):
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vectordb = Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        persist_directory=persist_dir
    )
    return vectordb


def load_or_create_db(csv_file, persistent_dir):
    if os.path.exists(persistent_dir):
        vectordb = Chroma(
            persist_directory=persistent_dir,
            embedding_function=OpenAIEmbeddings(model="text-embedding-3-small")
        )
    else:
        documents = load_csv_as_documents(csv_file)
        vectordb = build_vector_db(documents, persist_dir=persistent_dir)
    return vectordb


def search_db(query, vectordb):
    retriever = vectordb.as_retriever(search_kwargs={"k": 5})
    return retriever.invoke(query)
```

---

# ✅ 2. MCP Server — `adidas_rag_server.py`

This exposes a **tool named `search_adidas_products`**.

```python
import os
import asyncio
from mcp.server import Server
from mcp.types import Tool, ToolRequest, ToolResponse
from adidas_rag.rag_pipeline import load_or_create_db, search_db

server = Server("adidas_rag_server")

CSV_PATH = os.path.abspath("data/adidas.csv")
DB_PATH = "db/chroma_db_adidas"

vectordb = load_or_create_db(CSV_PATH, DB_PATH)


@server.tool(
    Tool(
        name="search_adidas_products",
        description="Search Adidas product dataset using vector similarity.",
        input_schema={
            "type": "object",
            "properties": {
                "query": {"type": "string"}
            },
            "required": ["query"]
        }
    )
)
async def search_products(request: ToolRequest) -> ToolResponse:
    query = request.params["query"]
    results = search_db(query, vectordb)

    output = []
    for doc in results:
        row = doc.page_content.split(" | ")
        kv = {}
        for item in row:
            if ": " in item:
                k, v = item.split(": ", 1)
                kv[k] = v
        output.append(kv)

    return ToolResponse(content=output)


if __name__ == "__main__":
    asyncio.run(server.run())
```

---

# ✅ 3. MCP Client/Agent — `adidas_rag_client.py`

This connects to the MCP server and queries the tool.

```python
import asyncio
from mcp.client import Client

async def main():
    client = Client("adidas_client")

    await client.connect("localhost", 8765)

    response = await client.call_tool(
        "search_adidas_products",
        {"query": "Which shoes have rating above 4.5?"}
    )

    print("\n=== RAG Search Results ===")
    for item in response.content:
        print(item)
        print("-" * 50)


if __name__ == "__main__":
    asyncio.run(main())
```

---

# ✅ 4. Running the System

## **Start the MCP Server**

```
python mcp/adidas_rag_server.py
```

You should see:

```
MCP Server started on port 8765
```

---

## **Run the MCP Client**

```
python mcp/adidas_rag_client.py
```

Output example:

```
=== RAG Search Results ===
{
  'name': 'Adidas Ultraboost',
  'color': 'Black',
  'category': 'Shoes',
  'average_rating': '4.7',
  ...
}
```

---

# 🎉 Now your RAG pipeline is fully MCP-compatible!

### This means any of the following can now call your tool:

* Claude Desktop
* VS Code MCP extension
* LangChain agents
* Web apps using MCP bridge
* Your own custom agent scripts

---

If you want, I can also generate:

✅ A LangChain Agent that uses your MCP server
✅ A Claude “Tool Plugin” YAML for direct integration
✅ A UI (React / Streamlit / FastAPI) for searching Adidas products
✅ Docker setup for server + client

Just tell me!
